# 🤖 Phase 4 — Feature Engineering & Modélisation
## Prédiction des Prix Immobiliers à Nouakchott, Mauritanie
### Master 1 Machine Learning — SupNum — Février 2026

---

> **Le problème :** En Mauritanie, il n'existe pas de base de données publique sur les prix de l'immobilier. Les prix sont négociés de gré à gré, les annonces sont en arabe hassaniya, et 72% des données sur les salles de bain sont manquantes.  
>  
> **Notre mission :** Construire un modèle capable de prédire le prix d'un bien à partir de 1153 annonces collectées sur voursa.com.  
>  
> **Résultat :** 🏆 **1ère place** sur la compétition Kaggle avec un RMSLE de **0.541**

---

## 📦 Setup

In [ ]:
import os, re, warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, r2_score
import xgboost as xgb
import joblib

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
RANDOM_STATE = 42

train = pd.read_csv('kaggle_train.csv')
test = pd.read_csv('kaggle_test.csv')
print(f'Train : {train.shape[0]} annonces × {train.shape[1]} colonnes')
print(f'Test  : {test.shape[0]} annonces à prédire')

---
# 1. 📊 Comprendre les données

Avant de modéliser, il faut comprendre ce qu'on a. Notre dataset contient **1153 annonces** immobilières de Nouakchott avec des textes en **arabe hassaniya** (dialecte mauritanien).

In [ ]:
# ── La variable cible : le prix ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution brute — très asymétrique
axes[0].hist(train['prix']/1e6, bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(train['prix'].mean()/1e6, color='red', linestyle='--', linewidth=2, label=f'Moyenne: {train["prix"].mean()/1e6:.1f}M')
axes[0].axvline(train['prix'].median()/1e6, color='green', linestyle='--', linewidth=2, label=f'Médiane: {train["prix"].median()/1e6:.1f}M')
axes[0].set_title('Distribution du Prix (brut)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Prix (millions MRU)')
axes[0].legend()

# Après log-transform — beaucoup plus symétrique
axes[1].hist(np.log1p(train['prix']), bins=50, color='coral', edgecolor='white', alpha=0.85)
axes[1].set_title('Après transformation log → quasi-normal', fontsize=13, fontweight='bold')
axes[1].set_xlabel('log(1 + Prix)')

plt.suptitle('Pourquoi on prédit en log-space ?', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Skewness brut  : {train["prix"].skew():.2f} (très asymétrique)')
print(f'Skewness log   : {np.log1p(train["prix"]).skew():.2f} (quasi-normal)')
print(f'\n→ La transformation log normalise la distribution et réduit l\'impact des extrêmes')

**💡 Interprétation :** La distribution brute est fortement asymétrique — quelques villas de luxe à Tevragh Zeina tirent la moyenne vers le haut. En travaillant en log-space, le modèle traite les écarts de manière proportionnelle (un écart de 50% est pénalisé pareil que le bien coûte 1M ou 10M).

In [ ]:
# ── Le facteur #1 : le quartier ────────────────────────────────────────────────
quartier_stats = train.groupby('quartier')['prix'].agg(['median', 'mean', 'count']).sort_values('median', ascending=True)

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#2ecc71' if m < 2e6 else '#f39c12' if m < 5e6 else '#e74c3c' for m in quartier_stats['median']]
quartier_stats['median'].plot(kind='barh', ax=ax, color=colors, edgecolor='white')

# Annotations
for i, (q, row) in enumerate(quartier_stats.iterrows()):
    ax.text(row['median'] + 100000, i, f'{row["median"]/1e6:.1f}M  (n={int(row["count"])})', 
            va='center', fontsize=10, fontweight='bold')

ax.set_title('Prix médian par quartier — Le facteur le plus discriminant', fontsize=14, fontweight='bold')
ax.set_xlabel('Prix médian (MRU)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
plt.tight_layout()
plt.show()

ratio = quartier_stats['median'].max() / quartier_stats['median'].min()
print(f'Le quartier le plus cher est {ratio:.1f}× plus cher que le moins cher')
print(f'Tevragh Zeina (ambassades, villas) vs Toujounine (périphérie)')

**💡 Interprétation :** Tevragh Zeina (quartier des ambassades) a un prix médian de **6.5M MRU** vs **1.1M** pour Toujounine. Un facteur **6×** ! Le quartier est de loin la variable la plus prédictive après la surface.

---
# 2. 🛠️ Feature Engineering — De 12 colonnes brutes à 45 features

C'est l'étape qui fait la différence. On transforme 12 colonnes brutes en **45 features informatives** que le modèle peut exploiter.

In [ ]:
# ══════════════════════════════════════════════════════════════
# FEATURE ENGINEERING COMPLET
# ══════════════════════════════════════════════════════════════
train['_split'] = 'train'; test['_split'] = 'test'
test_ids = test['id'].values
if 'prix' not in test.columns: test['prix'] = np.nan
combined = pd.concat([train, test], axis=0, ignore_index=True)

# --- Indicateurs de missingness ---
combined['nb_sdb_missing'] = combined['nb_sdb'].isna().astype(int)
combined['nb_chambres_missing'] = combined['nb_chambres'].isna().astype(int)

# --- Imputation ---
combined['nb_chambres'] = combined['nb_chambres'].fillna(combined['nb_chambres'].median())
combined['nb_salons'] = combined['nb_salons'].fillna(combined['nb_salons'].median())
combined['nb_sdb'] = combined['nb_sdb'].fillna(0)
combined['nb_salons'] = combined['nb_salons'].clip(upper=10)
combined['nb_chambres'] = combined['nb_chambres'].clip(upper=15)

# --- Temporel ---
combined['date_publication'] = pd.to_datetime(combined['date_publication'])
max_date = combined['date_publication'].max()
combined['days_since_pub'] = (max_date - combined['date_publication']).dt.days
combined['pub_month'] = combined['date_publication'].dt.month
combined['pub_dayofweek'] = combined['date_publication'].dt.dayofweek

# --- Caractéristiques structurées ---
combined['has_titre_foncier'] = combined['caracteristiques'].str.contains('Titre foncier', case=False, na=False).astype(int)
combined['has_garage'] = combined['caracteristiques'].str.contains('Garage', case=False, na=False).astype(int)
combined['has_camera'] = combined['caracteristiques'].str.contains('Caméra|Camera', case=False, na=False).astype(int)
combined['nb_balcons'] = combined['caracteristiques'].str.extract(r'(\d+)\s*balcon', expand=False).astype(float).fillna(0)
combined['taille_rue'] = combined['caracteristiques'].str.extract(r'Taille rue:\s*([\d.]+)', expand=False).astype(float).fillna(0)
combined['has_caracteristiques'] = (~combined['caracteristiques'].isna()).astype(int)
combined['nb_carac_pipes'] = combined['caracteristiques'].fillna('').str.count(r'\|')

# --- Type de bien (NLP arabe) ---
def extract_type_bien(titre):
    titre = str(titre)
    if re.search(r'فيلا|villa', titre, re.I): return 'villa'
    elif re.search(r'دوبلكس|ديبلكس|دبلكس|duplex|دوبليكس', titre, re.I): return 'duplex'
    elif re.search(r'آبرتمه|شقة|appartement|آبارتمه|ابارتمه', titre, re.I): return 'appartement'
    elif re.search(r'نيمرو|أرض|terrain|ارض', titre, re.I): return 'terrain'
    elif re.search(r'شانتيه|chantier', titre, re.I): return 'chantier'
    elif re.search(r'منزل', titre, re.I): return 'maison'
    elif re.search(r'دار', titre, re.I): return 'dar'
    else: return 'autre'
combined['type_bien'] = combined['titre'].apply(extract_type_bien)

# --- Mots-clés arabes ---
def count_keywords(text, keywords):
    text = str(text)
    return sum(1 for kw in keywords if kw in text)

for col in ['titre', 'description']:
    combined[f'{col}_luxury'] = combined[col].apply(lambda x: count_keywords(x, ['فاخر','لوكس','luxe','فخم','ممتاز','راقي']))
    combined[f'{col}_opportunity'] = combined[col].apply(lambda x: count_keywords(x, ['فرصة','فرصه','سمعه','سمعة']))
    combined[f'{col}_new'] = combined[col].apply(lambda x: count_keywords(x, ['جديد','جديدة','neuf','مجدد','اجديده']))
    combined[f'{col}_commercial'] = combined[col].apply(lambda x: count_keywords(x, ['تجاري','تجارية','محل','بوتيك']))
    combined[f'{col}_etage'] = combined[col].apply(lambda x: count_keywords(x, ['طابق','طابقين','étage']))

combined['desc_len'] = combined['description'].str.len().fillna(0)
combined['titre_len'] = combined['titre'].str.len().fillna(0)
combined['has_multiple_floors'] = combined['description'].str.contains('طابقين|طابق.*فوق', na=False).astype(int)

# --- Prix mentionné ---
def extract_price_from_text(text):
    text = str(text)
    m = re.findall(r'(\d+(?:[.,]\d+)?)\s*(?:مليون|ملايين|مليو)', text)
    if m:
        try:
            v = float(m[0].replace(',', '.'))
            if 0.1 <= v <= 200: return v * 1e6
        except: pass
    return 0
combined['prix_mentioned'] = combined['description'].apply(extract_price_from_text)
combined['prix_mentioned_any'] = np.maximum(combined['prix_mentioned'], combined['titre'].apply(extract_price_from_text))

# --- Interactions numériques ---
combined['log_surface'] = np.log1p(combined['surface_m2'])
combined['nb_pieces_total'] = combined['nb_chambres'] + combined['nb_salons'] + combined['nb_sdb']
combined['surface_par_piece'] = combined['surface_m2'] / (combined['nb_pieces_total'] + 1)
combined['chambres_x_surface'] = combined['nb_chambres'] * combined['surface_m2']
combined['surface_squared'] = combined['surface_m2'] ** 2
combined['sqrt_surface'] = np.sqrt(combined['surface_m2'])
combined['chambres_ratio'] = combined['nb_chambres'] / (combined['surface_m2'] + 1)

# --- Target Encoding KFold ---
combined['quartier_clean'] = combined['quartier'].astype(str).str.lower().str.strip()
train_mask = combined['_split'] == 'train'; test_mask = combined['_split'] == 'test'
gm = combined.loc[train_mask, 'prix'].mean(); gmed = combined.loc[train_mask, 'prix'].median()
combined['quartier_te_mean'] = gm; combined['quartier_te_median'] = gmed; combined['type_bien_te_mean'] = gm

ti = combined[train_mask].index.tolist()
kfe = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for tr, va in kfe.split(ti):
    atr = [ti[i] for i in tr]; ava = [ti[i] for i in va]
    combined.loc[ava, 'quartier_te_mean'] = combined.loc[ava, 'quartier_clean'].map(combined.loc[atr].groupby('quartier_clean')['prix'].mean()).fillna(gm).values
    combined.loc[ava, 'quartier_te_median'] = combined.loc[ava, 'quartier_clean'].map(combined.loc[atr].groupby('quartier_clean')['prix'].median()).fillna(gmed).values
    combined.loc[ava, 'type_bien_te_mean'] = combined.loc[ava, 'type_bien'].map(combined.loc[atr].groupby('type_bien')['prix'].mean()).fillna(gm).values

for c, t, a in [('quartier_clean','quartier_te_mean','mean'),('quartier_clean','quartier_te_median','median'),('type_bien','type_bien_te_mean','mean')]:
    mp = combined.loc[train_mask].groupby(c)['prix'].agg(a); fb = gm if a != 'median' else gmed
    combined.loc[test_mask, t] = combined.loc[test_mask, c].map(mp).fillna(fb).values

combined['surface_x_qte'] = combined['surface_m2'] * combined['quartier_te_mean'] / 1e6
combined['log_qte'] = np.log1p(combined['quartier_te_mean'])
combined['quartier_encoded'] = LabelEncoder().fit_transform(combined['quartier_clean'])
combined['type_bien_encoded'] = LabelEncoder().fit_transform(combined['type_bien'])

print(f'✅ Feature Engineering terminé : 12 colonnes brutes → {combined.shape[1]} colonnes')

### 2.1 Le NLP arabe — Extraire le type de bien du titre

Les annonces sont écrites en **arabe hassaniya** (dialecte mauritanien). Le titre contient souvent le type de bien, et ça change tout sur le prix.

In [ ]:
# ── Impact du type de bien sur le prix ─────────────────────────────────────────
train_part = combined[combined['_split'] == 'train']

type_stats = train_part.groupby('type_bien')['prix'].agg(['mean', 'median', 'count']).sort_values('mean', ascending=True)
type_stats = type_stats[type_stats['count'] >= 5]  # Au moins 5 annonces

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(type_stats)))[::-1]
bars = type_stats['mean'].plot(kind='barh', ax=ax, color=colors, edgecolor='white')

for i, (typ, row) in enumerate(type_stats.iterrows()):
    ax.text(row['mean'] + 100000, i, f'{row["mean"]/1e6:.1f}M (n={int(row["count"])})',
            va='center', fontsize=10, fontweight='bold')

ax.set_title('Prix moyen par type de bien — Extrait du titre en arabe', fontsize=14, fontweight='bold')
ax.set_xlabel('Prix moyen (MRU)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.0f}M'))
plt.tight_layout()
plt.show()

print('Exemples de titres et types extraits :')
examples = [
    ('فيلا فاخرة في تفرغ زينه', 'villa', '~9.4M'),
    ('ديبلكس للبيع في تفرغ زينه', 'duplex', '~6.2M'),
    ('منزل للبيع أفي عين الطلح', 'maison', '~5.1M'),
    ('دار فتيارت فاتح فبرك', 'dar', '~2.5M'),
    ('نيمرو للبيع في عرفات', 'terrain', '~1.8M'),
]
for titre, typ, prix in examples:
    print(f'  "{titre}" → {typ} ({prix} MRU)')

**💡 Interprétation :** فيلا (villa) = 9.4M en moyenne, دار (maison simple) = 2.5M. Un ratio de **3.7×** ! Cette seule feature, extraite par une regex sur le texte arabe, capture une information que les variables numériques seules ne peuvent pas donner.

### 2.2 Target Encoding — Encoder le quartier sans data leakage

> ⚠️ **Le piège du data leakage :** Si on encode le quartier avec la moyenne du prix calculée sur tout le train, le modèle "triche" car il voit indirectement la cible pendant l'entraînement.
>
> **Solution : KFold Target Encoding.** Chaque fold est encodé avec les statistiques des autres folds uniquement.

In [ ]:
# ── Schéma du KFold Target Encoding ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Gauche : Target Encoding naïf (LEAKAGE !)
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 6)
ax.set_title('❌ Target Encoding NAÏF = Data Leakage', fontsize=13, fontweight='bold', color='red')
ax.add_patch(plt.Rectangle((0.5, 3.5), 9, 2, fill=True, facecolor='#ffcccc', edgecolor='red', linewidth=2))
ax.text(5, 4.5, 'TOUT le train → calcul moyenne prix par quartier', ha='center', fontsize=11, fontweight='bold')
ax.annotate('', xy=(5, 3.2), xytext=(5, 3.5), arrowprops=dict(arrowstyle='->', color='red', lw=2))
ax.add_patch(plt.Rectangle((0.5, 1), 9, 2, fill=True, facecolor='#ffe0e0', edgecolor='red', linewidth=2))
ax.text(5, 2, 'Appliqué sur le MÊME train\n→ Le modèle voit la cible !', ha='center', fontsize=11)  # ✅ \n instead of real newline
ax.axis('off')

# Droite : KFold Target Encoding (CORRECT)
ax = axes[1]
ax.set_xlim(0, 10); ax.set_ylim(0, 6)
ax.set_title('✅ KFold Target Encoding = Pas de leakage', fontsize=13, fontweight='bold', color='green')
for i in range(5):
    y_pos = 5 - i * 0.8
    color = '#ccffcc' if i != 2 else '#ffffcc'
    label = f'Fold {i+1}' if i != 2 else 'Fold 3 (val)'
    ax.add_patch(plt.Rectangle((0.5, y_pos-0.3), 9, 0.6, fill=True, facecolor=color, edgecolor='green', linewidth=1))
    ax.text(5, y_pos, label, ha='center', fontsize=10, fontweight='bold' if i == 2 else 'normal')

ax.text(5, 0.5, 'Fold 3 encodé avec la moyenne des Folds 1,2,4,5', ha='center', fontsize=10, style='italic', color='green')
ax.axis('off')

plt.tight_layout()
plt.show()
print("Le KFold Target Encoding évite que le modèle voie la cible pendant l'entraînement")  # ✅ double quotes


### 2.3 Le prix mentionné — Une mine d'or dans les descriptions

Environ **50% des annonces** mentionnent directement le prix dans la description en arabe. Par exemple : "السعر 15 مليون" (le prix est 15 millions).

In [ ]:
# ── Prix mentionné vs prix réel ────────────────────────────────────────────────
has_prix = combined[(combined['_split'] == 'train') & (combined['prix_mentioned_any'] > 0)].copy()
no_prix = combined[(combined['_split'] == 'train') & (combined['prix_mentioned_any'] == 0)].copy()

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(has_prix['prix_mentioned_any']/1e6, has_prix['prix']/1e6, alpha=0.4, s=20, color='steelblue', label='Prix mentionné')
max_val = max(has_prix['prix'].max(), has_prix['prix_mentioned_any'].max()) / 1e6
ax.plot([0, max_val], [0, max_val], 'r--', linewidth=1, label='Ligne y=x (parfait)')
ax.set_xlabel('Prix mentionné dans la description (millions MRU)')
ax.set_ylabel('Prix réel (millions MRU)')
ax.set_title(f'Prix mentionné vs Prix réel ({len(has_prix)} annonces)', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Annonces avec prix mentionné : {len(has_prix)} ({100*len(has_prix)/len(combined[combined["_split"]=="train"]):.0f}%)')
print(f'Annonces sans prix mentionné : {len(no_prix)} ({100*len(no_prix)/len(combined[combined["_split"]=="train"]):.0f}%)')

**💡 Interprétation :** Quand le prix est mentionné, il est très corrélé au prix réel — c'est un signal fort pour le modèle. Pour les 50% d'annonces sans prix mentionné, le modèle se rabat sur les autres features.

### 2.4 Résumé : les 45 features

In [ ]:
# ── Catégories de features ─────────────────────────────────────────────────────
categories = {
    'Numériques brutes': ['surface_m2', 'nb_chambres', 'nb_salons', 'nb_sdb'],
    'Indicateurs manquants': ['nb_sdb_missing', 'nb_chambres_missing'],
    'Temporelles': ['days_since_pub', 'pub_month', 'pub_dayofweek'],
    'Caractéristiques': ['has_titre_foncier', 'has_garage', 'has_camera', 'nb_balcons', 'taille_rue', 'has_caracteristiques', 'nb_carac_pipes'],
    'NLP arabe': ['type_bien_encoded', 'desc_len', 'titre_len', 'titre_luxury', 'titre_opportunity', 'titre_new', 'titre_commercial', 'titre_etage', 'description_luxury', 'description_opportunity', 'description_new', 'description_commercial', 'description_etage', 'has_multiple_floors'],
    'Prix mentionné': ['prix_mentioned', 'prix_mentioned_any'],
    'Interactions': ['log_surface', 'nb_pieces_total', 'surface_par_piece', 'chambres_x_surface', 'surface_squared', 'sqrt_surface', 'chambres_ratio'],
    'Target Encoding': ['quartier_encoded', 'quartier_te_mean', 'quartier_te_median', 'type_bien_te_mean', 'surface_x_qte', 'log_qte'],
}

fig, ax = plt.subplots(figsize=(10, 5))
cat_counts = {k: len(v) for k, v in categories.items()}
colors = plt.cm.Set3(np.linspace(0, 1, len(cat_counts)))
bars = ax.barh(list(cat_counts.keys()), list(cat_counts.values()), color=colors, edgecolor='white')
for bar, count in zip(bars, cat_counts.values()):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2, str(count), va='center', fontweight='bold')
ax.set_title(f'45 features organisées en {len(categories)} catégories', fontsize=13, fontweight='bold')
ax.set_xlabel('Nombre de features')
plt.tight_layout()
plt.show()

total = sum(cat_counts.values())
print(f'Total : {total} features créées à partir de 12 colonnes brutes')

---
# 3. 🎯 Préparation des données

In [ ]:
feature_cols = [
    'surface_m2', 'nb_chambres', 'nb_salons', 'nb_sdb',
    'nb_sdb_missing', 'nb_chambres_missing',
    'days_since_pub', 'pub_month', 'pub_dayofweek',
    'has_titre_foncier', 'has_garage', 'has_camera',
    'nb_balcons', 'taille_rue', 'has_caracteristiques', 'nb_carac_pipes',
    'type_bien_encoded', 'desc_len', 'titre_len',
    'titre_luxury', 'titre_opportunity', 'titre_new', 'titre_commercial', 'titre_etage',
    'description_luxury', 'description_opportunity', 'description_new', 'description_commercial', 'description_etage',
    'prix_mentioned', 'prix_mentioned_any', 'has_multiple_floors',
    'log_surface', 'nb_pieces_total', 'surface_par_piece', 'chambres_x_surface',
    'surface_squared', 'sqrt_surface', 'chambres_ratio',
    'quartier_encoded', 'quartier_te_mean', 'quartier_te_median', 'type_bien_te_mean',
    'surface_x_qte', 'log_qte',
]

train_df = combined[combined['_split'] == 'train']
test_df = combined[combined['_split'] == 'test']

y = train_df['prix'].values
y_log = np.log1p(y)
X = train_df[feature_cols].astype(np.float64)
X_test = test_df[feature_cols].astype(np.float64)

print(f'{len(feature_cols)} features | X_train: {X.shape} | X_test: {X_test.shape}')

---
# 4. 🤖 Comparaison de 6 modèles

On teste du plus simple (régression linéaire) au plus avancé (XGBoost), tous évalués en **cross-validation 5-fold** pour une comparaison équitable.

$$RMSLE = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (\log(1+\hat{y}_i) - \log(1+y_i))^2}$$

In [ ]:
def rmsle(y_true, y_pred):
    return np.sqrt(np.mean((np.log1p(np.clip(y_true,0,None)) - np.log1p(np.clip(y_pred,0,None)))**2))

def evaluate_model(model, X, y_log, y_real, name=''):
    kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    oof = np.zeros(len(X))
    for _, (tri, vai) in enumerate(kf.split(X)):
        model.fit(X.values[tri], y_log[tri])
        oof[vai] = model.predict(X.values[vai])
    oof_prix = np.expm1(oof)
    return {
        'RMSLE': rmsle(y_real, oof_prix),
        'MAE': mean_absolute_error(y_real, oof_prix),
        'R²': r2_score(y_real, oof_prix),
        'oof': oof
    }

# ══════════════════════════════════════════════════════════════
# ENTRAÎNER LES 6 MODÈLES
# ══════════════════════════════════════════════════════════════
print('Entraînement de 6 modèles en 5-fold CV...')
print()

models = {
    '1. Régression Linéaire': LinearRegression(),
    '2. Ridge (α=10)': Ridge(alpha=10.0),
    '3. Lasso (α=0.01)': Lasso(alpha=0.01),
    '4. Random Forest': RandomForestRegressor(n_estimators=500, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1),
    '5. Gradient Boosting': GradientBoostingRegressor(n_estimators=500, learning_rate=0.05, max_depth=5, random_state=42),
    '6. XGBoost (tuned)': xgb.XGBRegressor(n_estimators=3000, learning_rate=0.02, max_depth=5, subsample=0.8, colsample_bytree=0.7, reg_alpha=0.3, reg_lambda=2.0, random_state=42, verbosity=0),
}

results = {}
for name, model in models.items():
    results[name] = evaluate_model(model, X, y_log, y, name)
    print(f'{name:30s} | RMSLE: {results[name]["RMSLE"]:.5f} | R²: {results[name]["R²"]:.4f} | MAE: {results[name]["MAE"]:>12,.0f} MRU')

print('\n✅ Tous les modèles entraînés')

In [ ]:
# ── Tableau comparatif visuel ──────────────────────────────────────────────────
comp = pd.DataFrame({n: {'RMSLE': r['RMSLE'], 'R²': r['R²'], 'MAE': r['MAE']} for n, r in results.items()}).T
comp = comp.sort_values('RMSLE')

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# RMSLE
colors_rmsle = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(comp))]
comp['RMSLE'].plot(kind='barh', ax=axes[0], color=colors_rmsle, edgecolor='white')
axes[0].set_title('RMSLE (↓ meilleur)', fontsize=13, fontweight='bold')
for i, v in enumerate(comp['RMSLE']):
    axes[0].text(v + 0.002, i, f'{v:.4f}', va='center', fontweight='bold')

# R²
colors_r2 = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(comp))]
comp['R²'].plot(kind='barh', ax=axes[1], color=colors_r2, edgecolor='white')
axes[1].set_title('R² (↑ meilleur)', fontsize=13, fontweight='bold')
for i, v in enumerate(comp['R²']):
    axes[1].text(v + 0.005, i, f'{v:.3f}', va='center', fontweight='bold')

# MAE
comp['MAE'].plot(kind='barh', ax=axes[2], color=colors_r2, edgecolor='white')
axes[2].set_title('MAE en MRU (↓ meilleur)', fontsize=13, fontweight='bold')
axes[2].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

plt.suptitle('Comparaison des 6 modèles — Cross-validation 5-fold', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'\n🏆 Meilleur modèle : {comp.index[0]}')
print(f'   RMSLE = {comp.iloc[0]["RMSLE"]:.5f}')
print(f'   R² = {comp.iloc[0]["R²"]:.4f}')

**💡 Interprétation :** 
- **Linéaire → XGBoost** : Le RMSLE passe de ~0.65 à ~0.58, soit une amélioration de **~11%**
- **R² : 0.55 → 0.75** : le modèle explique **75% de la variance** des prix
- **Les modèles linéaires** ne capturent pas l'interaction "grande surface à Tevragh Zeina >> grande surface à Arafat". Les arbres le font naturellement.
- **XGBoost avec depth=5** et forte régularisation (alpha=0.3, lambda=2.0) est le sweet spot pour ce petit dataset

---
# 5. 🔬 Analyse du meilleur modèle — XGBoost

### 5.1 Feature Importance — Quelles variables comptent le plus ?

In [ ]:
# ── Feature Importance ─────────────────────────────────────────────────────────
final_model = xgb.XGBRegressor(n_estimators=3000, learning_rate=0.02, max_depth=5,
    subsample=0.8, colsample_bytree=0.7, reg_alpha=0.3, reg_lambda=2.0, random_state=42, verbosity=0)
final_model.fit(X.values, y_log)

importances = pd.DataFrame({'feature': feature_cols, 'importance': final_model.feature_importances_})
importances = importances.sort_values('importance', ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(12, 10))
colors = ['#e74c3c' if imp > importances['importance'].quantile(0.8) else '#3498db' 
          for imp in importances['importance']]
importances.plot(kind='barh', x='feature', y='importance', ax=ax, color=colors, legend=False, edgecolor='white')
ax.set_title('Top 20 Features les plus importantes (XGBoost)', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

print('🏆 Top 5 features :')
for _, row in importances.tail(5).iloc[::-1].iterrows():
    print(f'  {row["feature"]:30s} {row["importance"]:.4f}')

**💡 Interprétation :** Le target encoding du quartier et la surface dominent. Le prix mentionné dans la description est aussi très informatif. Les features NLP (type de bien, mots-clés) apportent un signal complémentaire.

### 5.2 Analyse des erreurs — Où le modèle se trompe-t-il ?

In [ ]:
# ── Prédit vs Réel ─────────────────────────────────────────────────────────────
oof_preds = np.expm1(results['6. XGBoost (tuned)']['oof'])
errors_pct = np.abs(y - oof_preds) / y * 100

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# 1. Prédit vs Réel
axes[0].scatter(y/1e6, oof_preds/1e6, alpha=0.3, s=15, color='steelblue')
max_val = max(y.max(), oof_preds.max()) / 1e6
axes[0].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Parfait (y=x)')
axes[0].set_xlabel('Prix Réel (millions MRU)', fontsize=11)
axes[0].set_ylabel('Prix Prédit (millions MRU)', fontsize=11)
axes[0].set_title('Prédit vs Réel', fontsize=13, fontweight='bold')
axes[0].legend()

# 2. Distribution des erreurs
axes[1].hist(errors_pct[errors_pct < 100], bins=40, color='coral', edgecolor='white', alpha=0.85)
axes[1].axvline(np.median(errors_pct), color='red', linestyle='--', linewidth=2, label=f'Médiane: {np.median(errors_pct):.0f}%')
axes[1].set_title('Distribution des erreurs relatives', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Erreur relative (%)')
axes[1].legend()

# 3. Erreur par quartier
train_analysis = train_df.copy()
train_analysis['error_pct'] = errors_pct
error_by_q = train_analysis.groupby('quartier')['error_pct'].median().sort_values()
error_by_q.plot(kind='barh', ax=axes[2], color='steelblue', edgecolor='white')
axes[2].set_title('Erreur médiane par quartier', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Erreur relative médiane (%)')

plt.suptitle('Analyse des erreurs du modèle', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Erreur relative médiane : {np.median(errors_pct):.1f}%')
print(f'% prédictions à <20% erreur : {(errors_pct < 20).mean()*100:.0f}%')
print(f'% prédictions à <50% erreur : {(errors_pct < 50).mean()*100:.0f}%')

**💡 Interprétation :** Le modèle a une erreur médiane d'environ 25-30%. Les quartiers avec beaucoup de données (Tevragh Zeina, Teyarett) sont mieux prédits. Les quartiers avec peu de données (Sebkha, Riyadh) ont des erreurs plus élevées — c'est normal avec peu d'exemples d'entraînement.

---
# 6. 🏆 L'optimisation Kaggle — Comment on est passés 1ers

L'histoire de notre progression sur le leaderboard montre une leçon importante : **sur un petit dataset, la simplicité et la régularisation battent la complexité.**

In [ ]:
# ── Progression sur le leaderboard ──────────────────────────────────────────────
progression = pd.DataFrame({
    'Étape': [
        '1. Ensemble LGB+CB+XGB\n(3 modèles, Nelder-Mead)',
        '2. XGBoost seul\n(meilleur modèle individuel)',
        '3. XGB depth=5\n(réduire la profondeur)',
        '4. XGB depth=5 + hireg\n(α=0.3, λ=2.0)',
    ],
    'Score LB': [0.54909, 0.54701, 0.54471, 0.54174],
    'Rang': ['2ème', '2ème', '1er 🏆', '1er 🏆'],
})

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#e74c3c', '#f39c12', '#2ecc71', '#27ae60']
bars = ax.barh(range(len(progression)), progression['Score LB'], color=colors, edgecolor='white', height=0.6)

for i, (_, row) in enumerate(progression.iterrows()):
    ax.text(row['Score LB'] + 0.001, i, f'{row["Score LB"]:.5f} ({row["Rang"]})', 
            va='center', fontsize=12, fontweight='bold')

ax.set_yticks(range(len(progression)))
ax.set_yticklabels(progression['Étape'], fontsize=11)
ax.set_xlabel('Score RMSLE (↓ meilleur)', fontsize=12)
ax.set_title('Progression sur le Leaderboard Kaggle', fontsize=15, fontweight='bold')
ax.axvline(x=0.54620, color='gray', linestyle=':', linewidth=2, label='Score du 2ème (0.54620)')
ax.legend(fontsize=11)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('Leçons apprises :')
print('  1. L\'ensemble de 3 modèles était PIRE que XGBoost seul')
print('  2. Réduire la profondeur de 6 à 5 a amélioré → moins d\'overfit')
print('  3. Plus de régularisation (α=0.3, λ=2.0) a encore amélioré')
print('  → Sur un petit dataset (1153 lignes), la simplicité gagne')

**💡 Leçon clé :** On pense souvent que plus de modèles = meilleur score. En réalité, sur un dataset de 1153 lignes, un **seul XGBoost bien régularisé** bat un ensemble de 3 modèles. La raison : chaque modèle ajouté introduit sa propre variance d'erreur, et avec peu de données, cette variance ne se compense pas assez.

---
# 7. 💾 Prédictions & Sauvegarde du modèle

In [ ]:
# ── Prédictions multi-seeds pour Kaggle ────────────────────────────────────────
print('Entraînement multi-seeds...')
test_preds_all = []

for seed in [42, 123, 2024]:
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)
    tp = np.zeros(len(X_test))
    for _, (tri, vai) in enumerate(kf.split(X)):
        m = xgb.XGBRegressor(n_estimators=3000, learning_rate=0.02, max_depth=5,
            subsample=0.8, colsample_bytree=0.7, reg_alpha=0.3, reg_lambda=2.0,
            random_state=seed, verbosity=0, early_stopping_rounds=150)
        m.fit(X.values[tri], y_log[tri], eval_set=[(X.values[vai], y_log[vai])], verbose=False)
        tp += m.predict(X_test.values) / 5
    test_preds_all.append(tp)
    print(f'  Seed {seed} ✅')

final_preds = np.expm1(np.mean(test_preds_all, axis=0))
final_preds = np.clip(final_preds, 100_000, None)

print(f'\nPrédictions : mean={final_preds.mean():,.0f}, median={np.median(final_preds):,.0f}')

In [ ]:
# ── Sauvegarder ───────────────────────────────────────────────────────────────
# Submission Kaggle
submission = pd.DataFrame({'id': test_ids, 'prix': final_preds})
submission.to_csv('submission.csv', index=False)

# Modèle pour l'API (Phase 5)
joblib.dump(final_model, 'housing_model.pkl')
joblib.dump(feature_cols, 'features.pkl')

print('✅ Fichiers sauvegardés :')
print('  submission.csv — prédictions Kaggle')
print('  housing_model.pkl — modèle XGBoost pour l\'API')
print('  features.pkl — liste des 45 features')

In [ ]:
# ── Distribution finale ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(np.log1p(y), bins=50, alpha=0.5, color='steelblue', label='Train (réel)', edgecolor='white')
ax.hist(np.log1p(final_preds), bins=50, alpha=0.5, color='coral', label='Test (prédit)', edgecolor='white')
ax.set_xlabel('log(1 + prix)')
ax.set_title('Distributions similaires → le modèle prédit des prix réalistes', fontsize=13, fontweight='bold')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

---
# 📋 Résumé

| Aspect | Détail |
|--------|--------|
| **Dataset** | 1153 annonces, Nouakchott, texte en arabe hassaniya |
| **Features** | 45 (numériques, NLP arabe, target encoding KFold, interactions) |
| **Meilleur modèle** | XGBoost (depth=5, α=0.3, λ=2.0) |
| **CV RMSLE** | ~0.58 |
| **Score Kaggle** | **0.541 — 1ère place 🏆** |
| **R²** | 0.75 (explique 75% de la variance) |
| **Data leakage** | ✅ Aucun (KFold target encoding) |

**Les 3 facteurs clés de succès :**
1. Le **quartier** (target encoding) + la **surface** expliquent l'essentiel
2. Le **NLP arabe** (type de bien, prix mentionné) apporte un signal unique
3. La **régularisation forte** + profondeur faible évite l'overfit sur 1153 lignes

---
*🎓 Projet Capstone — Master 1 Machine Learning — SupNum 2026*